In [80]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
from langchain_core.runnables import RunnablePassthrough
from collections import Counter
import json


In [81]:
deepseek = ChatOpenAI(model="deepseek-chat",base_url = "https://api.deepseek.com", api_key = 'sk-xxxx', max_tokens=1024, temperature=0)
gpt4o = ChatOpenAI(model="gpt-4o", max_tokens=1024, temperature=1)
recommend_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "请给我推荐一本你认为比较好的书"),
    ]
)

recommend_chain = recommend_prompt | gpt4o | StrOutputParser()



In [82]:
book_name_prompt = ChatPromptTemplate.from_template(
     '请从下面这段文字中提取出其中的书名，用jsonArray的形式返回，比如["百年孤独","学会提问"]，其他任何内容都不要返回。 \n\n {content}'
)


In [83]:
composed_chain =  {"content":recommend_chain} | book_name_prompt | deepseek | JsonOutputParser()

In [84]:
composed_chain.invoke({})

['人类简史：从动物到上帝']

In [85]:
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

frequency_counter = Counter()
def invoke_with_catch():
    try:
        res = composed_chain.invoke({})
        return res
    except Exception as e:
        print(f"Exception occurred: {e}")
        return []

def main():

    with ThreadPoolExecutor(max_workers=10) as executor:  # 你可以调整 max_workers 的数量来控制线程数
        futures = [executor.submit(invoke_with_catch) for _ in range(1000)]
    
        for future in as_completed(futures):
            res = future.result()
            if res is not None:
                print(res)
                frequency_counter.update(res)

if __name__ == "__main__":
    main()

['人类简史：从动物到上帝']
['百年孤独', '枪炮、病菌与钢铁：人类社会的命运', '原则']
['百年孤独', '枪炮、病菌与钢铁：人类社会的命运', '原则']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['小王子']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['百年孤独', '枪炮、病菌与钢铁', '原则', '宇宙简史']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['百年孤独', '枪炮、病菌与钢铁：人类社会的命运', '原则']
['人类简史：从动物到上帝']
['百年孤独', '枪炮、病菌与钢铁：人类社会的命运', '高效能人士的七个习惯']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['百年孤独', '枪炮、病菌与钢铁', '原则']
['人类简史：从动物到上帝']
['百年孤独', '枪炮、病菌与钢铁：人类社会的命运', '原则']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['百年孤独', '枪炮、病菌与钢铁', '高效能人士的七个习惯']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从动物到上帝']
['人类简史：从

In [86]:
print("Initial counter:")
for string, count in frequency_counter.items():
    print(f"{string}: {count}")

Initial counter:
人类简史：从动物到上帝: 848
1984: 4
人类简史: 4
当下的力量: 1
百年孤独: 183
追风筝的人: 5
2001：太空漫游: 2
自控力: 1
枪炮、病菌与钢铁: 53
刀锋: 1
战争与和平: 2
1Q84: 1
杀死一只知更鸟: 3
当呼吸化为空气: 1
当尼采哭泣: 1
仿生人会梦见电子羊吗？: 1
史蒂夫·乔布斯传: 2
思考，快与慢: 6
高效能人士的七个习惯: 43
三体: 4
道德经: 1
局外人: 1
Sapiens: A Brief History of Humankind: 1
挪威的森林: 2
沙丘: 1
苏菲的世界: 2
如何赢得朋友和影响他人: 1
探险札记: 1
偷书贼: 1
基地: 1
管理的实践: 1
人类简史: 从动物到上帝: 1
影响力: 3
从优秀到卓越: 1
哈维尔·马里亚斯的世界: 1
小王子: 2
枪炮、病菌与钢铁：人类社会的命运: 107
原则: 118
宇宙简史: 22


In [88]:
sorted_counter = frequency_counter.most_common()

print(sorted_counter)

[('人类简史：从动物到上帝', 848), ('百年孤独', 183), ('原则', 118), ('枪炮、病菌与钢铁：人类社会的命运', 107), ('枪炮、病菌与钢铁', 53), ('高效能人士的七个习惯', 43), ('宇宙简史', 22), ('思考，快与慢', 6), ('追风筝的人', 5), ('1984', 4), ('人类简史', 4), ('三体', 4), ('杀死一只知更鸟', 3), ('影响力', 3), ('2001：太空漫游', 2), ('战争与和平', 2), ('史蒂夫·乔布斯传', 2), ('挪威的森林', 2), ('苏菲的世界', 2), ('小王子', 2), ('当下的力量', 1), ('自控力', 1), ('刀锋', 1), ('1Q84', 1), ('当呼吸化为空气', 1), ('当尼采哭泣', 1), ('仿生人会梦见电子羊吗？', 1), ('道德经', 1), ('局外人', 1), ('Sapiens: A Brief History of Humankind', 1), ('沙丘', 1), ('如何赢得朋友和影响他人', 1), ('探险札记', 1), ('偷书贼', 1), ('基地', 1), ('管理的实践', 1), ('人类简史: 从动物到上帝', 1), ('从优秀到卓越', 1), ('哈维尔·马里亚斯的世界', 1)]


### 